# LeetCode 1991 — Find the Middle Index in Array bằng Python

**Link đề:** https://leetcode.com/problems/find-the-middle-index-in-array/
**Độ khó:** Easy · **Chủ đề:** Array, Prefix Sum
*(Đề giống hệt LeetCode 724 — Find Pivot Index.)*

```python
class Solution:
    def findMiddleIndex(self, nums: List[int]) -> int:

```

## Mục tiêu học
1. Hiểu đề: tìm chỉ số **trái nhất** mà tổng bên trái bằng tổng bên phải (không tính chính nó).
2. Nắm `sum()`, slicing `nums[:i]`, `nums[i+1:]`, `enumerate`, và quy ước "tổng rỗng = 0".
3. Viết cách trực giác `O(n²)` rồi thấy công việc lặp lại.
4. Học kỹ thuật **prefix sum (tổng tiền tố)** và công thức `right = total - left - nums[i]`.
5. Trace, chứng minh đúng, phân tích `O(n)`, test đối chiếu và làm bài tập biến thể.

## 1. Đề bài yêu cầu gì?

Cho mảng `nums` (chỉ số từ 0). **middleIndex** là chỉ số `i` sao cho:

$$nums[0] + \dots + nums[i-1] \;=\; nums[i+1] + \dots + nums[n-1]$$

- Nếu `i = 0`, tổng bên trái coi là **0**. Nếu `i = n - 1`, tổng bên phải coi là **0**.
- Trả về middleIndex **trái nhất** (nhỏ nhất), hoặc `-1` nếu không có.

| `nums` | Kết quả | Giải thích |
|---|:-:|---|
| `[2,3,-1,8,4]` | `3` | trái: 2+3−1 = 4; phải: 4 |
| `[1,-1,4]` | `2` | trái: 1−1 = 0; phải: (rỗng) = 0 |
| `[2,5]` | `-1` | i=0: 0 vs 5; i=1: 2 vs 0 |

**Constraints:** `1 <= nums.length <= 100`, `-1000 <= nums[i] <= 1000` → **có số âm**, có thể có 0.

Chú ý: phần tử tại `i` **không** thuộc bên nào.

In [1]:
nums = [2, 3, -1, 8, 4]
i = 3
left_part, right_part = nums[:i], nums[i + 1:]
print("bên trái:", left_part, "tổng =", sum(left_part))
print("chính giữa:", nums[i])
print("bên phải:", right_part, "tổng =", sum(right_part))

bên trái: [2, 3, -1] tổng = 4
chính giữa: 8
bên phải: [4] tổng = 4


## 2. Nền tảng Python cần biết

### 2.1. Slicing `nums[:i]` và `nums[i+1:]`

- `nums[:i]` = các phần tử chỉ số `0 .. i-1` (bên trái `i`).
- `nums[i+1:]` = các phần tử chỉ số `i+1 .. n-1` (bên phải `i`).
- Khi đoạn rỗng, slicing trả `[]` — **không lỗi**.

In [2]:
nums = [1, -1, 4]
for i in range(len(nums)):
    print(f"i={i}: trái = {nums[:i]}, phải = {nums[i+1:]}")

i=0: trái = [], phải = [-1, 4]
i=1: trái = [1], phải = [4]
i=2: trái = [1, -1], phải = []


### 2.2. `sum()` và tổng rỗng

`sum([])` trả về `0`. Đây đúng là quy ước của đề: "nếu `i = 0` thì tổng trái là 0".

In [3]:
print(sum([2, 3, -1]))   # 4
print(sum([]))           # 0
print(sum([-5, 5]))      # 0 — số âm có thể triệt tiêu

4
0
0


### 2.3. `enumerate` — duyệt cả chỉ số lẫn giá trị

`for i, x in enumerate(nums)` cho ta cặp `(chỉ số, giá trị)` mỗi vòng, thay vì phải viết `nums[i]`.

In [4]:
for i, x in enumerate([2, 3, -1]):
    print("chỉ số", i, "giá trị", x)

chỉ số 0 giá trị 2
chỉ số 1 giá trị 3
chỉ số 2 giá trị -1


### 2.4. Tổng tiền tố (prefix sum) là gì?

`prefix[i]` = tổng của `i` phần tử đầu tiên = `nums[0] + ... + nums[i-1]`, với `prefix[0] = 0`.

Ví dụ `nums = [2, 3, -1, 8, 4]`:

| i | 0 | 1 | 2 | 3 | 4 | 5 |
|---|:-:|:-:|:-:|:-:|:-:|:-:|
| prefix[i] | 0 | 2 | 5 | 4 | 12 | 16 |

Công thức xây dựng: `prefix[i+1] = prefix[i] + nums[i]`. Khi có `prefix`, **tổng bất kỳ đoạn** `nums[l..r]` = `prefix[r+1] - prefix[l]` trong `O(1)`.

In [5]:
nums = [2, 3, -1, 8, 4]
prefix = [0]
for x in nums:
    prefix.append(prefix[-1] + x)      # prefix[-1] là phần tử cuối hiện tại
print("prefix =", prefix)

l, r = 1, 3
print(f"tổng nums[{l}..{r}] =", prefix[r + 1] - prefix[l], "| kiểm tra:", sum(nums[l:r + 1]))

prefix = [0, 2, 5, 4, 12, 16]
tổng nums[1..3] = 10 | kiểm tra: 10


### 2.5. `itertools.accumulate` — prefix sum có sẵn

`accumulate(nums)` sinh ra `nums[0], nums[0]+nums[1], ...`. Tham số `initial=0` (Python 3.8+) thêm số 0 ở đầu — đúng dạng `prefix` ở trên.

In [6]:
from itertools import accumulate
print(list(accumulate([2, 3, -1, 8, 4])))
print(list(accumulate([2, 3, -1, 8, 4], initial=0)))

[2, 5, 4, 12, 16]
[0, 2, 5, 4, 12, 16]


## 3. Cách 1 — Trực giác: với mỗi `i`, cộng lại hai bên

Ý tưởng: duyệt `i` từ trái sang phải (để gặp chỉ số **trái nhất** trước). Với mỗi `i` tính `sum(nums[:i])` và `sum(nums[i+1:])`. Bằng nhau → trả `i` ngay.

In [7]:
def middle_index_bruteforce(nums):
    for i in range(len(nums)):
        if sum(nums[:i]) == sum(nums[i + 1:]):
            return i
    return -1

print(middle_index_bruteforce([2, 3, -1, 8, 4]))  # 3
print(middle_index_bruteforce([1, -1, 4]))        # 2
print(middle_index_bruteforce([2, 5]))            # -1

3
2
-1


### Code nộp LeetCode — Cách 1

In [8]:
from typing import List

class SolutionBrute:
    def findMiddleIndex(self, nums: List[int]) -> int:
        for i in range(len(nums)):
            if sum(nums[:i]) == sum(nums[i + 1:]):
                return i
        return -1

print(SolutionBrute().findMiddleIndex([2, 3, -1, 8, 4]))

3


### Nút thắt của Cách 1

Mỗi `i` cộng lại gần như **toàn bộ mảng** (`n − 1` phần tử) → tổng `O(n²)`. Với `n ≤ 100` vẫn qua, nhưng lãng phí:

- Bước `i` cộng `nums[0..i-1]`; bước `i+1` lại cộng `nums[0..i]` — **lặp lại** `i` phép cộng cũ, chỉ thêm `nums[i]`.
- Bên phải cũng vậy: chỉ **bớt** đi một phần tử.

Câu hỏi: có thể **cập nhật** tổng trái/phải thay vì tính lại?

In [9]:
def count_additions(n):
    # số phép cộng mà cách 1 thực hiện (xấp xỉ: mỗi i cộng n-1 phần tử)
    return sum((i) + (n - i - 1) for i in range(n))

for n in [5, 100, 10_000]:
    print(f"n = {n:>6}: cách 1 cộng ~{count_additions(n):,} phần tử")

n =      5: cách 1 cộng ~20 phần tử
n =    100: cách 1 cộng ~9,900 phần tử
n =  10000: cách 1 cộng ~99,990,000 phần tử


## 4. Insight — Tổng phải = Tổng toàn bộ − Tổng trái − phần tử giữa

Gọi `total = sum(nums)` và `left` = tổng bên trái `i`. Vì mảng chia thành 3 phần (trái, `nums[i]`, phải):

$$right = total - left - nums[i]$$

Điều kiện middle index `left == right` trở thành:

$$left == total - left - nums[i] \iff 2 \cdot left + nums[i] == total$$

Và khi đi từ `i` sang `i+1`, chỉ cần `left += nums[i]`. Mỗi bước `O(1)`!

In [10]:
nums = [2, 3, -1, 8, 4]
total = sum(nums)
print("total =", total)
left = 0
for i, x in enumerate(nums):
    right = total - left - x
    print(f"i={i}: left = {left:>2}, nums[i] = {x:>2}, right = {total} - {left} - {x} = {right}")
    left += x

total = 16
i=0: left =  0, nums[i] =  2, right = 16 - 0 - 2 = 14
i=1: left =  2, nums[i] =  3, right = 16 - 2 - 3 = 11
i=2: left =  5, nums[i] = -1, right = 16 - 5 - -1 = 12
i=3: left =  4, nums[i] =  8, right = 16 - 4 - 8 = 4
i=4: left = 12, nums[i] =  4, right = 16 - 12 - 4 = 0


## 5. Cách 2 — Mảng prefix sum

Trước khi tối ưu bộ nhớ, viết bản dùng mảng `prefix` (đúng như hint 3):
- `left = prefix[i]` (tổng `i` phần tử đầu).
- `right = prefix[n] - prefix[i+1]` (tổng từ `i+1` tới hết).

In [11]:
def middle_index_prefix(nums):
    n = len(nums)
    prefix = [0] * (n + 1)          # list n+1 số 0
    for i in range(n):
        prefix[i + 1] = prefix[i] + nums[i]
    for i in range(n):
        left = prefix[i]
        right = prefix[n] - prefix[i + 1]
        if left == right:
            return i
    return -1

print(middle_index_prefix([2, 3, -1, 8, 4]))
print(middle_index_prefix([1, -1, 4]))
print(middle_index_prefix([2, 5]))

3
2
-1


`[0] * (n + 1)` tạo list gồm `n + 1` số 0 — cách nhanh để cấp sẵn chỗ rồi gán theo chỉ số.

## 6. Cách 3 — Một biến `left` + `total` (lời giải chính)

Giảng từng đoạn:

```python
total = sum(nums)            # (1) tổng toàn mảng, tính 1 lần
left = 0                     # (2) tổng bên trái của i = 0 là rỗng
for i, x in enumerate(nums): # (3) đi từ trái -> gặp chỉ số trái nhất trước
    if left == total - left - x:   # (4) so trái với phải
        return i
    left += x                # (5) chuẩn bị cho i+1: x chuyển sang bên trái
return -1                    # (6) không có chỉ số nào thỏa
```

**Thứ tự (4) rồi (5) rất quan trọng:** khi kiểm tra `i`, `left` chưa được cộng `nums[i]` (phần tử giữa không thuộc bên trái).

In [12]:
def middle_index(nums):
    total = sum(nums)
    left = 0
    for i, x in enumerate(nums):
        if left == total - left - x:
            return i
        left += x
    return -1

print(middle_index([2, 3, -1, 8, 4]))  # 3
print(middle_index([1, -1, 4]))        # 2
print(middle_index([2, 5]))            # -1

3
2
-1


## 7. TRACE từng bước

Ví dụ `nums = [2, 3, -1, 8, 4]`, `total = 16`.

| i | x | left (trước) | right = 16 − left − x | left == right? | left (sau) |
|:-:|:-:|:-:|:-:|:-:|:-:|
| 0 | 2 | 0 | 14 | không | 2 |
| 1 | 3 | 2 | 11 | không | 5 |
| 2 | −1 | 5 | 12 | không | 4 |
| 3 | 8 | 4 | 4 | **có → return 3** | — |

Ví dụ thứ hai `[1, -1, 4]` cho thấy nhánh **biên phải** (bên phải rỗng ở `i = n−1`).

In [13]:
def trace_middle_index(nums):
    total = sum(nums)
    print(f"nums = {nums}, total = {total}")
    left = 0
    for i, x in enumerate(nums):
        right = total - left - x
        ok = left == right
        print(f"  i={i}, x={x:>2} | left={left:>3}, right={right:>3} | {'BẰNG -> return ' + str(i) if ok else 'khác'}")
        if ok:
            return i
        left += x
    print("  hết mảng -> return -1")
    return -1

trace_middle_index([2, 3, -1, 8, 4])
print()
trace_middle_index([1, -1, 4])
print()
trace_middle_index([2, 5])

nums = [2, 3, -1, 8, 4], total = 16
  i=0, x= 2 | left=  0, right= 14 | khác
  i=1, x= 3 | left=  2, right= 11 | khác
  i=2, x=-1 | left=  5, right= 12 | khác
  i=3, x= 8 | left=  4, right=  4 | BẰNG -> return 3

nums = [1, -1, 4], total = 4
  i=0, x= 1 | left=  0, right=  3 | khác
  i=1, x=-1 | left=  1, right=  4 | khác
  i=2, x= 4 | left=  0, right=  0 | BẰNG -> return 2

nums = [2, 5], total = 7
  i=0, x= 2 | left=  0, right=  5 | khác
  i=1, x= 5 | left=  2, right=  0 | khác
  hết mảng -> return -1


-1

### Nhánh đặc biệt: nhiều middle index — phải lấy trái nhất

`nums = [0, 0, 0]`: mọi chỉ số đều thỏa (0 = 0). Vì ta `return` **ngay lần đầu** khi duyệt từ trái, kết quả là `0`.

Thêm một ví dụ có số âm `[-1, 1, 5, -1, 1]`: tổng trái đi qua giá trị âm rồi mới khớp ở `i = 2`.

In [14]:
trace_middle_index([0, 0, 0])
print()
trace_middle_index([-1, 1, 5, -1, 1])

nums = [0, 0, 0], total = 0
  i=0, x= 0 | left=  0, right=  0 | BẰNG -> return 0

nums = [-1, 1, 5, -1, 1], total = 5
  i=0, x=-1 | left=  0, right=  6 | khác
  i=1, x= 1 | left= -1, right=  5 | khác
  i=2, x= 5 | left=  0, right=  0 | BẰNG -> return 2


2

## 8. Code nộp LeetCode

In [15]:
class Solution:
    def findMiddleIndex(self, nums: List[int]) -> int:
        total = sum(nums)
        left = 0
        for i, x in enumerate(nums):
            if 2 * left + x == total:
                return i
            left += x
        return -1

for nums, expected in [([2, 3, -1, 8, 4], 3), ([1, -1, 4], 2), ([2, 5], -1)]:
    got = Solution().findMiddleIndex(nums)
    print(nums, "->", got)
    assert got == expected

[2, 3, -1, 8, 4] -> 3
[1, -1, 4] -> 2
[2, 5] -> -1


Bản nộp dùng dạng tương đương `2 * left + x == total` (đã biến đổi ở mục 4) — ít phép tính hơn một chút, kết quả y hệt.

## 9. Cách 4 — Hai biến `left`, `right` chạy song song

Biến thể khác của cùng ý tưởng: khởi tạo `right = total`, mỗi bước **trừ** `x` khỏi `right` trước khi so, rồi **cộng** `x` vào `left` sau khi so. Cách này rất trực quan: phần tử `x` "đi" từ bên phải, qua chính giữa, sang bên trái.

In [16]:
def middle_index_two_vars(nums):
    left, right = 0, sum(nums)
    for i, x in enumerate(nums):
        right -= x               # x rời bên phải, đứng ở giữa
        if left == right:
            return i
        left += x                # x sang bên trái
    return -1

for nums in ([2, 3, -1, 8, 4], [1, -1, 4], [2, 5], [0, 0, 0]):
    print(nums, "->", middle_index_two_vars(nums))

[2, 3, -1, 8, 4] -> 3
[1, -1, 4] -> 2
[2, 5] -> -1
[0, 0, 0] -> 0


## 10. Vì sao đúng? (invariant)

**Invariant:** tại đầu vòng lặp ứng với chỉ số `i`, `left = nums[0] + ... + nums[i-1]`.

- **Khởi tạo** (`i = 0`): `left = 0` = tổng rỗng. ✓
- **Duy trì:** cuối vòng `i` ta làm `left += nums[i]` → đầu vòng `i+1`, `left` = tổng `0..i`. ✓
- Khi đó `total − left − nums[i]` đúng bằng tổng `nums[i+1..n-1]` vì ba phần (trái, giữa, phải) phủ kín mảng và không chồng nhau.

**Trái nhất:** ta duyệt `i` tăng dần và `return` ở lần thỏa đầu tiên, nên không có chỉ số thỏa nào nhỏ hơn bị bỏ qua. Nếu duyệt hết mà không thỏa → không tồn tại → `-1`.

Chú ý: vì có **số âm**, ta **không** thể dừng sớm kiểu "left đã lớn hơn right thì thôi" — tổng có thể giảm trở lại.

In [17]:
# Ví dụ cho thấy left > right ở giữa chừng nhưng sau đó vẫn bằng nhau
trace_middle_index([2, 1, -3, 5, -5, 0])

nums = [2, 1, -3, 5, -5, 0], total = 0
  i=0, x= 2 | left=  0, right= -2 | khác
  i=1, x= 1 | left=  2, right= -3 | khác
  i=2, x=-3 | left=  3, right=  0 | khác
  i=3, x= 5 | left=  0, right= -5 | khác
  i=4, x=-5 | left=  5, right=  0 | khác
  i=5, x= 0 | left=  0, right=  0 | BẰNG -> return 5


5

## 11. Time / Space và bảng so sánh

Gọi `n = len(nums)`.

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Cộng lại hai bên | `sum(nums[:i])`, `sum(nums[i+1:])` | `O(n²)` | `O(n)` (slice tạm) | Dễ nghĩ nhất |
| 2. Mảng prefix | `prefix[i]`, `prefix[n] - prefix[i+1]` | `O(n)` | `O(n)` | Tổng quát cho truy vấn đoạn |
| 3. `left` + `total` | `2*left + x == total` | `O(n)` | `O(1)` | **Khuyến nghị** |
| 4. `left`, `right` song song | trừ phải, so, cộng trái | `O(n)` | `O(1)` | Rất trực quan |

Cách 3/4 duyệt mảng **2 lần** (một lần `sum`, một lần vòng for) — vẫn là `O(n)`.

## 12. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

Nhóm test:
1. Ba ví dụ đề.
2. Một phần tử: `[5]` → `0` (cả hai bên rỗng).
3. Biên trái: `[0, 1, -1]` → `0`; biên phải: `[1, -1, 0]` → `2`.
4. Nhiều đáp án: `[0, 0, 0]` → `0`.
5. Số âm, toàn số âm.
6. Đối chiếu ngẫu nhiên có seed với Cách 1.

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<28} PASS ({actual})")

check("đề 1", middle_index([2, 3, -1, 8, 4]), 3)
check("đề 2", middle_index([1, -1, 4]), 2)
check("đề 3", middle_index([2, 5]), -1)
check("một phần tử", middle_index([5]), 0)
check("biên trái", middle_index([0, 1, -1]), 0)
check("biên phải", middle_index([1, -1, 0]), 2)
check("nhiều đáp án -> trái nhất", middle_index([0, 0, 0]), 0)
check("toàn số âm", middle_index([-1, -2, -3, -3]), 2)
check("không có", middle_index([1, 2, 3]), -1)

đề 1                         PASS (3)
đề 2                         PASS (2)
đề 3                         PASS (-1)
một phần tử                  PASS (0)
biên trái                    PASS (0)
biên phải                    PASS (2)
nhiều đáp án -> trái nhất    PASS (0)
toàn số âm                   PASS (2)
không có                     PASS (-1)


In [19]:
import random

rng = random.Random(1991)
for _ in range(3000):
    nums = [rng.randint(-3, 3) for _ in range(rng.randint(1, 10))]
    expected = middle_index_bruteforce(nums)
    assert middle_index_prefix(nums) == expected
    assert middle_index(nums) == expected
    assert middle_index_two_vars(nums) == expected
    assert Solution().findMiddleIndex(nums) == expected
print("3000 test ngẫu nhiên: 4 cách khớp với brute force")

3000 test ngẫu nhiên: 4 cách khớp với brute force


Ta chọn giá trị nhỏ `[-3, 3]` để tổng **hay bằng nhau** — nhờ đó các nhánh "tìm thấy" và "nhiều đáp án" được thử thường xuyên, không chỉ nhánh `-1`.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Cộng `nums[i]` vào `left` **trước** khi so sánh

In [20]:
def wrong_add_first(nums):
    total = sum(nums)
    left = 0
    for i, x in enumerate(nums):
        left += x                       # cộng quá sớm: x bị tính vào bên trái
        if left == total - left - x:
            return i
    return -1

print("sai :", wrong_add_first([2, 3, -1, 8, 4]))   # -1
print("đúng:", middle_index([2, 3, -1, 8, 4]))      # 3

sai : -1
đúng: 3


**Vì sao sai:** lúc so sánh, `left` đã chứa phần tử giữa. **Sửa:** so trước, cộng sau (hoặc dùng mô hình 2 biến ở Cách 4).

### Lỗi 2: Quên trừ phần tử giữa: `right = total - left`

In [21]:
def wrong_forget_middle(nums):
    total = sum(nums)
    left = 0
    for i, x in enumerate(nums):
        if left == total - left:        # thiếu "- x"
            return i
        left += x
    return -1

print("sai :", wrong_forget_middle([2, 3, -1, 8, 4]))
print("đúng:", middle_index([2, 3, -1, 8, 4]))

sai : -1
đúng: 3


**Vì sao sai:** `total - left` là tổng từ `i` trở đi — **đã bao gồm** `nums[i]`. **Sửa:** `total - left - nums[i]`.

### Lỗi 3: Duyệt từ phải sang trái → trả chỉ số **phải nhất**

In [22]:
def wrong_rightmost(nums):
    total = sum(nums)
    right = 0
    for i in range(len(nums) - 1, -1, -1):
        if right == total - right - nums[i]:
            return i
        right += nums[i]
    return -1

print("sai :", wrong_rightmost([0, 0, 0]))   # 2
print("đúng:", middle_index([0, 0, 0]))      # 0

sai : 2
đúng: 0


**Vì sao sai:** đề yêu cầu **leftmost**. **Sửa:** duyệt từ trái, trả ngay lần đầu.

### Lỗi 4: Dừng sớm khi `left > right`
Với số âm, tổng không đơn điệu (xem ví dụ `[2, 1, -3, 5, -5, 0]` ở mục 10). Không được "cắt tỉa" kiểu này.

## 14. Mẹo nhận dạng pattern

- "Tổng bên trái / bên phải", "chia mảng thành hai phần có tổng bằng nhau", "tổng đoạn `[l, r]`" → **prefix sum**.
- Công thức vàng: **`right = total − left − giữa`** — chỉ cần một biến chạy.
- "Trả về chỉ số **trái nhất**" → duyệt trái sang phải và `return` ngay.
- Có số âm → đừng giả định tổng tăng dần; đừng dùng hai con trỏ kiểu thu hẹp.

## 15. Biến thể

1. **LeetCode 724 — Find Pivot Index**: giống hệt.
2. **Trả về tất cả middle index.**
3. **LeetCode 1013 — chia thành 3 phần tổng bằng nhau.**
4. **Truy vấn tổng đoạn nhiều lần** (LeetCode 303 — Range Sum Query): dựng `prefix` một lần, mỗi truy vấn `O(1)`.

In [23]:
def all_middle_indices(nums):
    total, left, res = sum(nums), 0, []
    for i, x in enumerate(nums):
        if 2 * left + x == total:
            res.append(i)
        left += x
    return res

class NumArray:                     # LeetCode 303
    def __init__(self, nums):
        self.prefix = [0]
        for x in nums:
            self.prefix.append(self.prefix[-1] + x)
    def sumRange(self, left, right):
        return self.prefix[right + 1] - self.prefix[left]

print("tất cả middle index [0,0,0]:", all_middle_indices([0, 0, 0]))
na = NumArray([-2, 0, 3, -5, 2, -1])
print("sumRange(0,2) =", na.sumRange(0, 2), "| sumRange(2,5) =", na.sumRange(2, 5))

tất cả middle index [0,0,0]: [0, 1, 2]
sumRange(0,2) = 1 | sumRange(2,5) = -1


## 16. Bài tập tự luyện (TODO — chưa chạy)

### Bài 1 — Tính tay
Lập bảng `i, x, left, right` cho `nums = [1, 7, 3, 6, 5, 6]`. Middle index là bao nhiêu?

### Bài 2 — Dự đoán output
`middle_index([2, -2, 7])` trả về gì? Vì sao không phải `-1`?

### Bài 3 — Tự viết lời giải chính

In [24]:
def exercise_middle_index(nums: list[int]) -> int:
    # TODO 1: total = tổng mảng, left = 0
    # TODO 2: duyệt i, x; so left với total - left - x
    # TODO 3: cộng x vào left SAU khi so
    pass

# assert exercise_middle_index([2, 3, -1, 8, 4]) == 3
# assert exercise_middle_index([2, 5]) == -1

### Bài 4 — Biến thể: tìm chỉ số `i` (0 ≤ i < n−1) chia mảng thành `nums[:i+1]` và `nums[i+1:]` có tổng bằng nhau (phần tử `i` thuộc bên trái). Trả về `i` nhỏ nhất hoặc `-1`.

In [25]:
def exercise_split_equal(nums: list[int]) -> int:
    # TODO: left bao gồm nums[i]; right = total - left; phải còn ít nhất 1 phần tử bên phải
    pass

# assert exercise_split_equal([1, 2, 3, 3, 2, 1]) == 2
# assert exercise_split_equal([1, 1, 1]) == -1

### Bài 5 — Xây prefix sum và trả lời nhiều truy vấn

In [26]:
def exercise_range_sums(nums: list[int], queries: list[tuple[int, int]]) -> list[int]:
    # TODO: dựng prefix một lần, mỗi truy vấn (l, r) trả prefix[r+1] - prefix[l]
    pass

# assert exercise_range_sums([1, 2, 3, 4], [(0, 1), (1, 3), (2, 2)]) == [3, 9, 3]

### Bài 6 — Mini challenge: chỉ số `i` làm `|left − right|` **nhỏ nhất** (hòa thì lấy trái nhất)

In [27]:
def exercise_most_balanced(nums: list[int]) -> int:
    # TODO: như lời giải chính, nhưng theo dõi chênh lệch nhỏ nhất thay vì return ngay
    pass

# assert exercise_most_balanced([2, 5]) == 1          # |0-5|=5, |2-0|=2
# assert exercise_most_balanced([2, 3, -1, 8, 4]) == 3

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | total = 28 | left chạy 0, 1, 8, 11 | tại i=3: left 11, right 28−11−6 = 11 |
| 2 | Tổng âm có thể triệt tiêu | total = 7 | i=2: left = 0, right = 0 |
| 3 | Khuôn mục 6 | `if left == total - left - x` | `left += x` sau `if` |
| 4 | `left` gồm cả `nums[i]` | `range(len(nums) - 1)` | `if 2 * left == total` |
| 5 | `prefix = [0]` | `prefix.append(prefix[-1] + x)` | list comprehension trên queries |
| 6 | Theo dõi `best_diff`, `best_i` | `abs(left - right)` | chỉ cập nhật khi `<` (không `<=`) |

## 18. Đáp án tham khảo, template và checklist

- Bài 1: total = 28; i=0: 0 vs 27; i=1: 1 vs 20; i=2: 8 vs 17; i=3: **11 vs 11** → **3**.
- Bài 2: total = 7; i=0: 0 vs 5; i=1: 2 vs 7; i=2: **0 vs 0** → **2** (vì 2 + (−2) = 0, bên phải rỗng = 0).
- Bài 6 với `[2, 5]`: i=0 → |0 − 5| = 5; i=1 → |2 − 0| = 2 → đáp án **1**.

In [28]:
def answer_middle_index(nums):
    total, left = sum(nums), 0
    for i, x in enumerate(nums):
        if left == total - left - x:
            return i
        left += x
    return -1

def answer_split_equal(nums):
    total, left = sum(nums), 0
    for i in range(len(nums) - 1):
        left += nums[i]
        if 2 * left == total:
            return i
    return -1

def answer_range_sums(nums, queries):
    prefix = [0]
    for x in nums:
        prefix.append(prefix[-1] + x)
    return [prefix[r + 1] - prefix[l] for l, r in queries]

def answer_most_balanced(nums):
    total, left = sum(nums), 0
    best_i, best_diff = -1, float('inf')
    for i, x in enumerate(nums):
        d = abs(left - (total - left - x))
        if d < best_diff:
            best_i, best_diff = i, d
        left += x
    return best_i

assert answer_middle_index([1, 7, 3, 6, 5, 6]) == 3
assert answer_middle_index([2, -2, 7]) == 2
assert answer_middle_index([2, 3, -1, 8, 4]) == 3 and answer_middle_index([2, 5]) == -1
assert answer_split_equal([1, 2, 3, 3, 2, 1]) == 2 and answer_split_equal([1, 1, 1]) == -1
assert answer_range_sums([1, 2, 3, 4], [(0, 1), (1, 3), (2, 2)]) == [3, 9, 3]
assert answer_most_balanced([2, 5]) == 1
assert answer_most_balanced([2, 3, -1, 8, 4]) == 3
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


### Template nhớ nhanh

```python
class Solution:
    def findMiddleIndex(self, nums: List[int]) -> int:
        total, left = sum(nums), 0
        for i, x in enumerate(nums):
            if 2 * left + x == total:   # left == total - left - x
                return i
            left += x                    # so trước, cộng sau
        return -1
```

Câu thần chú: **"Phải = Tổng − Trái − Giữa."**

### Checklist trước khi submit
- [ ] So sánh **trước**, cộng `x` vào `left` **sau**?
- [ ] Đã trừ phần tử giữa khi tính bên phải?
- [ ] Duyệt từ trái và `return` ngay để lấy chỉ số **trái nhất**?
- [ ] Không dừng sớm dựa trên `left > right` (có số âm)?
- [ ] Đã test mảng 1 phần tử và trường hợp biên phải (`[1, -1, 4]`)?